# Additional Baselines for NeurIPS Submission

**Run on Google Colab with GPU runtime.**

Baselines:
1. MC Dropout (on DistMult)
2. Deep Ensembles (5x DistMult)
3. Temperature Scaling

All compared on FB15k-237 for OOD detection.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import roc_auc_score
import json
import os
import random
import urllib.request

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [2]:
CONFIG = {
    'epochs': 50,
    'embedding_dim': 100,
    'batch_size': 2048,
    'lr': 0.001,
    'dropout': 0.3,
    'mc_samples': 20,
    'n_ensembles': 5,
    'seeds': [42, 123, 456],
}

In [3]:
# Download FB15k-237
def download_fb15k237():
    os.makedirs('data', exist_ok=True)
    base_url = "https://raw.githubusercontent.com/DeepGraphLearning/KnowledgeGraphEmbedding/master/data/FB15k-237"

    for split in ['train', 'test']:
        path = f'data/{split}.txt'
        if not os.path.exists(path):
            print(f"Downloading {split}...")
            urllib.request.urlretrieve(f"{base_url}/{split}.txt", path)
    print("Download complete!")

download_fb15k237()

def load_triples(path):
    triples = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                triples.append((parts[0], parts[1], parts[2]))
    return triples

train = load_triples('data/train.txt')
test = load_triples('data/test.txt')

entities = set()
relations = set()
for h, r, t in train + test:
    entities.add(h)
    entities.add(t)
    relations.add(r)

print(f"FB15k-237: {len(train)} train, {len(test)} test")
print(f"Entities: {len(entities)}, Relations: {len(relations)}")

ent2idx = {e: i for i, e in enumerate(entities)}
rel2idx = {r: i for i, r in enumerate(relations)}

Download complete!
FB15k-237: 272115 train, 20466 test
Entities: 14534, Relations: 237


## Baseline 1: MC Dropout

In [4]:
class DistMultDropout(nn.Module):
    """DistMult with dropout for MC Dropout uncertainty."""
    def __init__(self, num_entities, num_relations, dim, dropout=0.3):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, heads, relations, tails):
        h = self.dropout(self.entity_emb(heads))
        r = self.relation_emb(relations)
        t = self.dropout(self.entity_emb(tails))
        return (h * r * t).sum(dim=-1)

    def get_uncertainty_mc(self, heads, relations, tails, n_samples=20):
        """MC Dropout uncertainty: variance of predictions."""
        self.train()  # Enable dropout
        scores = []
        with torch.no_grad():
            for _ in range(n_samples):
                score = torch.sigmoid(self.forward(heads, relations, tails))
                scores.append(score)
        scores = torch.stack(scores)
        # Predictive entropy as uncertainty
        mean_score = scores.mean(dim=0)
        entropy = -mean_score * torch.log(mean_score + 1e-10) - (1-mean_score) * torch.log(1-mean_score + 1e-10)
        return entropy

## Baseline 2: Deep Ensembles

In [5]:
class DistMult(nn.Module):
    """Standard DistMult."""
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, heads, relations, tails):
        h = self.entity_emb(heads)
        r = self.relation_emb(relations)
        t = self.entity_emb(tails)
        return (h * r * t).sum(dim=-1)


class DeepEnsemble:
    """Ensemble of DistMult models."""
    def __init__(self, models):
        self.models = models

    def get_uncertainty(self, heads, relations, tails):
        """Ensemble uncertainty: variance of predictions."""
        scores = []
        for model in self.models:
            model.eval()
            with torch.no_grad():
                score = torch.sigmoid(model(heads, relations, tails))
                scores.append(score)
        scores = torch.stack(scores)
        # Predictive entropy
        mean_score = scores.mean(dim=0)
        entropy = -mean_score * torch.log(mean_score + 1e-10) - (1-mean_score) * torch.log(1-mean_score + 1e-10)
        return entropy

## Baseline 3: Coverage-Only (Our Strong Baseline)

In [6]:
class CoverageOnly:
    def __init__(self, num_entities, num_relations):
        self.coverage = torch.zeros(num_entities, num_relations)

    def fit(self, triples, ent2idx, rel2idx):
        for h, r, t in triples:
            self.coverage[ent2idx[h], rel2idx[r]] = 1.0
            self.coverage[ent2idx[t], rel2idx[r]] = 1.0
        return self

    def get_uncertainty(self, heads, relations, tails):
        h_seen = self.coverage[heads.cpu(), relations.cpu()]
        t_seen = self.coverage[tails.cpu(), relations.cpu()]
        return (2.0 - h_seen - t_seen).to(heads.device)

## Training Functions

In [7]:
def train_model(model, triples, ent2idx, rel2idx, epochs):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])
    criterion = nn.BCEWithLogitsLoss()

    heads = torch.tensor([ent2idx[h] for h, r, t in triples])
    relations = torch.tensor([rel2idx[r] for h, r, t in triples])
    tails = torch.tensor([ent2idx[t] for h, r, t in triples])

    loader = DataLoader(
        TensorDataset(heads, relations, tails),
        batch_size=CONFIG['batch_size'], shuffle=True
    )

    model.train()
    for epoch in range(epochs):
        for batch_h, batch_r, batch_t in loader:
            batch_h = batch_h.to(device)
            batch_r = batch_r.to(device)
            batch_t = batch_t.to(device)

            pos = model(batch_h, batch_r, batch_t)
            neg_t = torch.randint(0, len(ent2idx), batch_t.shape, device=device)
            neg = model(batch_h, batch_r, neg_t)

            loss = criterion(pos, torch.ones_like(pos)) + criterion(neg, torch.zeros_like(neg))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}")

    return model


def evaluate_auroc(get_uncertainty_fn, test, ent2idx, rel2idx):
    heads = torch.tensor([ent2idx.get(h, 0) for h, r, t in test]).to(device)
    relations = torch.tensor([rel2idx.get(r, 0) for h, r, t in test]).to(device)
    tails = torch.tensor([ent2idx.get(t, 0) for h, r, t in test]).to(device)

    with torch.no_grad():
        id_unc = get_uncertainty_fn(heads, relations, tails).cpu().numpy()
        neg_tails = torch.randint(0, len(ent2idx), tails.shape, device=device)
        ood_unc = get_uncertainty_fn(heads, relations, neg_tails).cpu().numpy()

    labels = np.concatenate([np.ones(len(id_unc)), np.zeros(len(ood_unc))])
    scores = np.concatenate([-id_unc, -ood_unc])
    return roc_auc_score(labels, scores)

## Run All Baselines

In [8]:
results = {
    'CoverageOnly': [],
    'MCDropout': [],
    'DeepEnsemble': [],
}

for seed in CONFIG['seeds']:
    print(f"\n{'='*50}")
    print(f"Seed {seed}")
    print('='*50)

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # 1. Coverage Only
    print("\n1. Coverage Only...")
    cov = CoverageOnly(len(ent2idx), len(rel2idx))
    cov.fit(train, ent2idx, rel2idx)
    auroc = evaluate_auroc(cov.get_uncertainty, test, ent2idx, rel2idx)
    results['CoverageOnly'].append(auroc)
    print(f"   AUROC: {auroc:.4f}")

    # 2. MC Dropout
    print("\n2. MC Dropout...")
    mc_model = DistMultDropout(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'], CONFIG['dropout'])
    mc_model = train_model(mc_model, train, ent2idx, rel2idx, CONFIG['epochs'])
    auroc = evaluate_auroc(
        lambda h, r, t: mc_model.get_uncertainty_mc(h, r, t, CONFIG['mc_samples']),
        test, ent2idx, rel2idx
    )
    results['MCDropout'].append(auroc)
    print(f"   AUROC: {auroc:.4f}")

    # 3. Deep Ensemble
    print("\n3. Deep Ensemble...")
    ensemble_models = []
    for i in range(CONFIG['n_ensembles']):
        torch.manual_seed(seed + i * 1000)
        model = DistMult(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
        model = train_model(model, train, ent2idx, rel2idx, CONFIG['epochs'])
        ensemble_models.append(model)
        print(f"   Trained model {i+1}/{CONFIG['n_ensembles']}")
    ensemble = DeepEnsemble(ensemble_models)
    auroc = evaluate_auroc(ensemble.get_uncertainty, test, ent2idx, rel2idx)
    results['DeepEnsemble'].append(auroc)
    print(f"   AUROC: {auroc:.4f}")


Seed 42

1. Coverage Only...
   AUROC: 0.8189

2. MC Dropout...
  Epoch 10/50
  Epoch 20/50
  Epoch 30/50
  Epoch 40/50
  Epoch 50/50
   AUROC: 0.4230

3. Deep Ensemble...
  Epoch 10/50
  Epoch 20/50
  Epoch 30/50
  Epoch 40/50
  Epoch 50/50
   Trained model 1/5
  Epoch 10/50
  Epoch 20/50
  Epoch 30/50
  Epoch 40/50
  Epoch 50/50
   Trained model 2/5
  Epoch 10/50
  Epoch 20/50
  Epoch 30/50
  Epoch 40/50
  Epoch 50/50
   Trained model 3/5
  Epoch 10/50
  Epoch 20/50
  Epoch 30/50
  Epoch 40/50
  Epoch 50/50
   Trained model 4/5
  Epoch 10/50
  Epoch 20/50
  Epoch 30/50
  Epoch 40/50
  Epoch 50/50
   Trained model 5/5
   AUROC: 0.2232

Seed 123

1. Coverage Only...
   AUROC: 0.8208

2. MC Dropout...
  Epoch 10/50
  Epoch 20/50
  Epoch 30/50
  Epoch 40/50
  Epoch 50/50
   AUROC: 0.4266

3. Deep Ensemble...
  Epoch 10/50
  Epoch 20/50
  Epoch 30/50
  Epoch 40/50
  Epoch 50/50
   Trained model 1/5
  Epoch 10/50
  Epoch 20/50
  Epoch 30/50
  Epoch 40/50
  Epoch 50/50
   Trained model 2/5

In [9]:
# Compare with CAGP (from previous experiments)
CAGP_RESULT = {'mean': 0.960, 'std': 0.000}  # FB15k-237

print("\n" + "="*70)
print("FB15k-237 BASELINE COMPARISON")
print("="*70)
print(f"{'Method':<20} {'AUROC':<15} {'vs CAGP'}")
print("-"*50)

for method in results:
    mean = np.mean(results[method])
    std = np.std(results[method])
    delta = mean - CAGP_RESULT['mean']
    print(f"{method:<20} {mean:.4f} ± {std:.3f}   {delta:+.4f}")

print(f"{'CAGP (ours)':<20} {CAGP_RESULT['mean']:.4f} ± {CAGP_RESULT['std']:.3f}   +0.0000")
print("-"*50)

print("\nConclusion:")
best_baseline = max(np.mean(results[m]) for m in results)
gap = CAGP_RESULT['mean'] - best_baseline
if gap > 0:
    print(f"CAGP beats best baseline by {gap:.4f} ({gap/best_baseline*100:.1f}%)")
else:
    print(f"Best baseline beats CAGP by {-gap:.4f}")


FB15k-237 BASELINE COMPARISON
Method               AUROC           vs CAGP
--------------------------------------------------
CoverageOnly         0.8206 ± 0.001   -0.1394
MCDropout            0.4298 ± 0.007   -0.5302
DeepEnsemble         0.2245 ± 0.001   -0.7355
CAGP (ours)          0.9600 ± 0.000   +0.0000
--------------------------------------------------

Conclusion:
CAGP beats best baseline by 0.1394 (17.0%)


In [10]:
# Save results
output = {
    'dataset': 'FB15k-237',
    'config': CONFIG,
    'results': {
        m: {'mean': float(np.mean(results[m])), 'std': float(np.std(results[m]))}
        for m in results
    },
    'cagp_reference': CAGP_RESULT
}

with open('baseline_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("\nResults saved to baseline_results.json")
print(json.dumps(output, indent=2))


Results saved to baseline_results.json
{
  "dataset": "FB15k-237",
  "config": {
    "epochs": 50,
    "embedding_dim": 100,
    "batch_size": 2048,
    "lr": 0.001,
    "dropout": 0.3,
    "mc_samples": 20,
    "n_ensembles": 5,
    "seeds": [
      42,
      123,
      456
    ]
  },
  "results": {
    "CoverageOnly": {
      "mean": 0.8205774683402249,
      "std": 0.0012947896358988247
    },
    "MCDropout": {
      "mean": 0.42976842682854866,
      "std": 0.007183872207924426
    },
    "DeepEnsemble": {
      "mean": 0.22446047772270442,
      "std": 0.0010466648271216998
    }
  },
  "cagp_reference": {
    "mean": 0.96,
    "std": 0.0
  }
}
